# 🤖 Smart Document Classifier — EDA & Training Notebook

**For 3rd-year CS/DS students | Fully offline | No cloud APIs**

This notebook walks you through:
1. 📊 Exploratory Data Analysis (EDA)
2. 🔤 TF-IDF feature extraction + visualization
3. 🏋️ Model training (TF-IDF + LogReg)
4. 📈 Evaluation + confusion matrix
5. 🧠 Sentence embeddings + UMAP visualization
6. 🚀 Stretch: DistilBERT fine-tuning

**Run on:** Local Jupyter OR Google Colab (free GPU for DistilBERT)

In [ ]:
# ── Install dependencies (Colab) ────────────────────────────────────────────
# Uncomment if running on Colab:
# !pip install scikit-learn sentence-transformers umap-learn seaborn matplotlib pandas

import os, sys, json, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
warnings.filterwarnings('ignore')

# Adjust path for project structure
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
DATA_PATH = os.path.join(PROJECT_ROOT, 'data', 'processed', 'classification.csv')
MODELS_DIR = os.path.join(PROJECT_ROOT, 'ml', 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

CLASSES = ['resume', 'invoice', 'research_paper', 'lab_report', 'college_notes']
COLORS = ['#3b82f6', '#10b981', '#8b5cf6', '#f59e0b', '#ef4444']
CLASS_COLORS = dict(zip(CLASSES, COLORS))

print('✅ Setup complete')

## 📊 Step 1: Generate Training Data

Run `generate_samples.py` first to create training data, OR use this cell:

In [ ]:
# Generate samples if they don't exist
if not os.path.exists(DATA_PATH):
    gen_script = os.path.join(PROJECT_ROOT, 'data', 'generate_samples.py')
    os.system(f'python {gen_script}')

df = pd.read_csv(DATA_PATH)
df = df[df['label'].isin(CLASSES)].dropna(subset=['text', 'label'])
df['text_len'] = df['text'].str.len()

print(f'📦 Dataset shape: {df.shape}')
print(f'\n📈 Class distribution:')
print(df['label'].value_counts())

In [ ]:
# ── EDA Plots ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Smart Document Classifier — EDA', fontsize=14, fontweight='bold')

# Class distribution
counts = df['label'].value_counts()
axes[0].bar(counts.index, counts.values, color=[CLASS_COLORS[c] for c in counts.index])
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=20)

# Text length by class
df.boxplot(column='text_len', by='label', ax=axes[1])
axes[1].set_title('Text Length by Class')
axes[1].set_xlabel('Class')
axes[1].set_ylabel('Characters')
axes[1].tick_params(axis='x', rotation=20)
plt.sca(axes[1])
plt.title('')

# Text length histogram
for cls in CLASSES:
    subset = df[df['label'] == cls]['text_len']
    axes[2].hist(subset, bins=20, alpha=0.5, label=cls, color=CLASS_COLORS[cls])
axes[2].set_title('Text Length Distribution')
axes[2].set_xlabel('Characters')
axes[2].set_ylabel('Frequency')
axes[2].legend(fontsize=7)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'eda_plots.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 EDA plots saved!')

## 🔤 Step 2: TF-IDF Feature Extraction

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
import scipy.sparse as sp

X = df['text'].values
y = df['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

tfidf = TfidfVectorizer(
    max_features=20000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=2, max_df=0.95,
)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print(f'TF-IDF matrix shape (train): {X_train_tfidf.shape}')
print(f'Vocabulary size: {len(tfidf.vocabulary_):,} terms')
print(f'Matrix density: {X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]) * 100:.3f}%')

In [ ]:
# ── Top TF-IDF terms per class ───────────────────────────────────────────────
feature_names = np.array(tfidf.get_feature_names_out())

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
fig.suptitle('Top 15 TF-IDF Terms Per Class', fontsize=13, fontweight='bold')

for i, cls in enumerate(CLASSES):
    cls_mask = y_train == cls
    if cls_mask.sum() == 0: continue
    mean_tfidf = X_train_tfidf[cls_mask].mean(axis=0).A1
    top_idx = mean_tfidf.argsort()[-15:][::-1]
    
    axes[i].barh(feature_names[top_idx][::-1], mean_tfidf[top_idx][::-1], color=CLASS_COLORS[cls])
    axes[i].set_title(cls.replace('_', ' ').title(), fontsize=10, color=CLASS_COLORS[cls], fontweight='bold')
    axes[i].tick_params(axis='y', labelsize=8)
    axes[i].set_xlabel('Mean TF-IDF', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'tfidf_terms.png'), dpi=150, bbox_inches='tight')
plt.show()

## 🏋️ Step 3: Train Classifier

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = LogisticRegression(
    C=5.0, max_iter=1000,
    multi_class='multinomial', solver='lbfgs',
    random_state=42, class_weight='balanced'
)
clf.fit(X_train_tfidf, y_train)

y_pred = clf.predict(X_test_tfidf)
y_proba = clf.predict_proba(X_test_tfidf)

acc = accuracy_score(y_test, y_pred)
print(f'🎯 Test Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'\n{classification_report(y_test, y_pred, target_names=CLASSES, zero_division=0)}')

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred, labels=CLASSES)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title(f'Confusion Matrix (Accuracy: {acc*100:.1f}%)', fontsize=12)
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('True')
axes[0].tick_params(axis='x', rotation=25)

# Normalized
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[1], vmin=0, vmax=1)
axes[1].set_title('Normalized Confusion Matrix', fontsize=12)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('True')
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Save model ────────────────────────────────────────────────────────────────
with open(os.path.join(MODELS_DIR, 'tfidf.pkl'), 'wb') as f:
    pickle.dump(tfidf, f)
with open(os.path.join(MODELS_DIR, 'classifier.pkl'), 'wb') as f:
    pickle.dump(clf, f)

meta = {
    'model_type': 'tfidf_logreg', 'classes': CLASSES,
    'test_accuracy': round(acc, 4), 'vocab_size': len(tfidf.vocabulary_),
    'train_samples': len(X_train), 'test_samples': len(X_test),
}
with open(os.path.join(MODELS_DIR, 'metadata.json'), 'w') as f:
    json.dump(meta, f, indent=2)

print(f'✅ Models saved!')
print(f'   models/tfidf.pkl')
print(f'   models/classifier.pkl')

## 🧠 Step 4: Sentence Embeddings + UMAP Visualization

In [ ]:
# This requires: pip install sentence-transformers umap-learn
try:
    from sentence_transformers import SentenceTransformer
    import umap

    print('Loading all-MiniLM-L6-v2 (~80MB, one-time download)...')
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Encode a sample (50 per class for speed)
    sample_df = df.groupby('label').head(50)
    embeddings = model.encode(sample_df['text'].str[:512].tolist(), show_progress_bar=True)
    labels_sample = sample_df['label'].values

    print(f'Embeddings shape: {embeddings.shape}')

    # UMAP 2D reduction
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    emb_2d = reducer.fit_transform(embeddings)

    # Plot
    fig, ax = plt.subplots(figsize=(10, 7))
    for cls in CLASSES:
        mask = labels_sample == cls
        ax.scatter(emb_2d[mask, 0], emb_2d[mask, 1],
                   label=cls, color=CLASS_COLORS[cls], alpha=0.7, s=60, edgecolors='white', linewidth=0.5)

    ax.set_title('UMAP: Sentence Embeddings (all-MiniLM-L6-v2)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10, markerscale=1.2)
    ax.set_xlabel('UMAP-1')
    ax.set_ylabel('UMAP-2')
    ax.grid(True, alpha=0.2)

    plt.tight_layout()
    plt.savefig(os.path.join(MODELS_DIR, 'umap_embeddings.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('📊 UMAP visualization saved!')

except ImportError:
    print('⚠️  Install: pip install sentence-transformers umap-learn')
    print('   Skipping embedding visualization.')

## 🚀 Step 5 (Stretch): DistilBERT Fine-tuning

Uncomment and run on Google Colab with GPU for best results.
Expected accuracy: **95%+** after 3 epochs.

In [ ]:
# STRETCH GOAL — DistilBERT Fine-tuning
# Requires: pip install transformers torch accelerate datasets
# GPU recommended! (~15 min on Colab T4)

RUN_DISTILBERT = False  # Set True to run

if RUN_DISTILBERT:
    from transformers import DistilBertTokenizer, DistilBertForSequenceClassification, TrainingArguments, Trainer
    from torch.utils.data import Dataset
    import torch

    label2id = {cls: i for i, cls in enumerate(CLASSES)}
    id2label = {i: cls for i, cls in enumerate(CLASSES)}

    class DocDataset(Dataset):
        def __init__(self, texts, labels, tokenizer, label2id):
            self.enc = tokenizer(list(texts), max_length=256, padding='max_length',
                                  truncation=True, return_tensors='pt')
            self.labels = torch.tensor([label2id[l] for l in labels], dtype=torch.long)
        def __len__(self): return len(self.labels)
        def __getitem__(self, i):
            return {k: v[i] for k, v in self.enc.items()} | {'labels': self.labels[i]}

    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    model = DistilBertForSequenceClassification.from_pretrained(
        'distilbert-base-uncased', num_labels=len(CLASSES),
        id2label=id2label, label2id=label2id
    )

    train_ds = DocDataset(X_train, y_train, tokenizer, label2id)
    test_ds  = DocDataset(X_test, y_test, tokenizer, label2id)

    args = TrainingArguments(
        output_dir=os.path.join(MODELS_DIR, 'distilbert'),
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        evaluation_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        warmup_ratio=0.1,
        logging_steps=10,
    )

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds)
    trainer.train()

    results = trainer.evaluate()
    print(f'\n🎯 DistilBERT Eval Loss: {results["eval_loss"]:.4f}')

    # Save
    bert_dir = os.path.join(MODELS_DIR, 'distilbert')
    model.save_pretrained(bert_dir)
    tokenizer.save_pretrained(bert_dir)
    print(f'✅ DistilBERT saved to {bert_dir}/')
else:
    print('Set RUN_DISTILBERT = True to run fine-tuning (needs GPU)')

## ✅ Summary

| Step | Tool | Purpose |
|------|------|---------|
| OCR | Tesseract | Text extraction from PDF/image |
| Vectorizer | TF-IDF (20k features, bigrams) | Represent documents as sparse vectors |
| Classifier | LogisticRegression (C=5) | Predict document class |
| Embeddings | all-MiniLM-L6-v2 (384-dim) | Dense semantic vectors |
| Similarity | Cosine similarity | Find related documents |
| (Stretch) | DistilBERT fine-tune | Higher accuracy with GPU |

**Next:** Run `node backend/server.js` and `npm run dev` in frontend/ to test the full pipeline!